# Analyse Client - Instacart

**Plan du notebook**

| Section | Contenu |
|---|---|
| 0 | Setup (imports, chemins) |
| 1 | Chargement raisonné des 5 fichiers |
| 2 | EDA & contrôle qualité |
| 3 | Table de travail (jointures) |
| 4–8 | Une section par question |
| 9 | Synthèse |

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path("Datasets")

pd.set_option("display.max_columns", None)   # ne jamais tronquer les colonnes à l'affichage
sns.set_theme(style="whitegrid")             # style de graphique par défaut

print("pandas", pd.__version__)

pandas 3.0.5


## 1. Chargement raisonné

`order_products.csv` fait **32 millions de lignes**. Si on laisse pandas deviner les types, il met des `int64`
partout (8 octets par nombre) et le texte en `object` : on gaspille ~3× la mémoire nécessaire.

La bonne pratique : **imposer le type de chaque colonne à la lecture** avec `dtype=`.

Règles utilisées ici :
- `int8` : entiers de −128 à 127 → `order_dow` (0–6), `department_id` (1–21), `reordered` (0/1)
- `int16` : jusqu'à ~32 000 → `order_number`, `add_to_cart_order`, `aisle_id`
- `int32` : jusqu'à ~2 milliards → tous les `*_id` volumineux (`order_id`, `user_id`, `product_id`)
- `float32` : `days_since_prior_order` (a des valeurs manquantes → doit être un float)
- `category` : `eval_set` (seulement 3 valeurs distinctes répétées 3,4 M de fois)
- `string` : les libellés texte

Note : `order_hour_of_day` est écrit `"08"` dans le fichier ; demander `int8` suffit, pandas convertit `"08" → 8`.

### 1.1 Les 3 dictionnaires (petits fichiers)

In [2]:
aisles = pd.read_csv(DATA / "aisles.csv", dtype={
    "aisle_id": "int16",
    "aisle": "string",
})

departments = pd.read_csv(DATA / "departments.csv", dtype={
    "department_id": "int8",
    "department": "string",
})

products = pd.read_csv(DATA / "products.csv", dtype={
    "product_id": "int32",
    "product_name": "string",
    "aisle_id": "int16",
    "department_id": "int8",
})

print("aisles     ", aisles.shape)
print("departments", departments.shape)
print("products   ", products.shape)
products.head()

aisles      (134, 2)
departments (21, 2)
products    (49688, 4)


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


### 1.2 Les commandes (`orders`)

In [3]:
orders = pd.read_csv(DATA / "orders.csv", dtype={
    "order_id": "int32",
    "user_id": "int32",
    "eval_set": "category",
    "order_number": "int16",
    "order_dow": "int8",
    "order_hour_of_day": "int8",
    "days_since_prior_order": "float32",
})

print(orders.shape)
print(orders["eval_set"].value_counts())
orders.head()

(3421083, 7)
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


### 1.3 Le contenu des paniers (`order_products`)

Le gros fichier. Avec les `dtype` ci-dessous : 4 + 4 + 2 + 1 = **11 octets/ligne** → ~350 Mo en RAM.
Sans typage, ce serait ~1 Go. Le chargement peut quelques dizaines de secondes.

In [4]:
order_products = pd.read_csv(DATA / "order_products.csv", dtype={
    "order_id": "int32",
    "product_id": "int32",
    "add_to_cart_order": "int16",
    "reordered": "int8",
})

print(order_products.shape)
order_products.head()

(32434489, 4)


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


### 1.4 Vérification de l'empreinte mémoire

In [5]:
for nom, df in [("aisles", aisles), ("departments", departments), ("products", products),
                ("orders", orders), ("order_products", order_products)]:
    mo = df.memory_usage(deep=True).sum() / 1e6
    print(f"{nom:16} {df.shape[0]:>10,} lignes   {mo:8.1f} Mo")

aisles                  134 lignes        0.0 Mo
departments              21 lignes        0.0 Mo
products             49,688 lignes        4.3 Mo
orders            3,421,083 lignes       58.2 Mo
order_products   32,434,489 lignes      356.8 Mo


## 2. EDA & contrôle qualité

Avant de répondre à une seule question, on vérifie que les données se comportent comme on l'imagine. Trois
familles de vérifications :

1. **Valeurs manquantes** : où, combien, est-ce logique ?
2. **Doublons** : une clé censée être unique (`order_id`, `product_id`) l'est-elle vraiment ?
3. **Cohérence entre tables** : est-ce que les jointures qu'on prévoit de faire vont bien matcher ?

C'est l'étape qu'on est tenté de sauter — et où se cachent les surprises qui faussent tout le reste.

### 2.1 Valeurs manquantes

In [6]:
for nom, df in [("aisles", aisles), ("departments", departments), ("products", products),
                ("orders", orders), ("order_products", order_products)]:
    na = df.isna().sum()
    na = na[na > 0]
    print(f"--- {nom} ---")
    print(na if len(na) else "aucune valeur manquante")
    print()

--- aisles ---
aucune valeur manquante

--- departments ---
aucune valeur manquante

--- products ---
aucune valeur manquante

--- orders ---
days_since_prior_order    206209
dtype: int64

--- order_products ---
aucune valeur manquante



Seule `orders.days_since_prior_order` a des manquants : **206 209**. C'est exactement le nombre de clients
distincts (`user_id`). L'hypothèse à vérifier : ce sont les 1ʳᵉˢ commandes de chaque client (`order_number == 1`),
pour lesquelles il n'existe pas de "commande précédente" — donc le NaN est **normal**, pas une erreur de saisie.

In [7]:
print("n° de clients distincts       :", orders["user_id"].nunique())
print("lignes avec délai manquant    :", orders["days_since_prior_order"].isna().sum())
print("1res commandes (order_number==1):", (orders["order_number"] == 1).sum())

# vérification : le manquant coïncide-t-il EXACTEMENT avec order_number == 1 ?
premiere_commande = orders["order_number"] == 1
delai_manquant = orders["days_since_prior_order"].isna()
print("coïncidence parfaite :", (premiere_commande == delai_manquant).all())

n° de clients distincts       : 206209
lignes avec délai manquant    : 206209
1res commandes (order_number==1): 206209
coïncidence parfaite : True


### 2.2 Doublons sur les clés d'identité

`order_id` doit être unique dans `orders`, `product_id` unique dans `products`. On vérifie.

In [8]:
print("order_id dupliqués dans orders     :", orders["order_id"].duplicated().sum())
print("product_id dupliqués dans products :", products["product_id"].duplicated().sum())
print("aisle_id dupliqués dans aisles     :", aisles["aisle_id"].duplicated().sum())
print("department_id dupliqués            :", departments["department_id"].duplicated().sum())

order_id dupliqués dans orders     : 0
product_id dupliqués dans products : 0
aisle_id dupliqués dans aisles     : 0
department_id dupliqués            : 0


### 2.3 Cohérence entre `orders` et `order_products` — le point le plus important de cette section

Jusqu'ici je t'ai dit qu'`eval_set` valait `prior` / `train` / `test`, et que `test` n'avait pas de contenu de
panier. Vérifions ce qu'il en est **réellement** pour `train` aussi, au lieu de le supposer : quels `eval_set`
sont effectivement présents dans `order_products` ?

In [9]:
ids_avec_panier = order_products["order_id"].unique()
print("order_id distincts dans order_products :", len(ids_avec_panier))

couverture = orders.loc[orders["order_id"].isin(ids_avec_panier), "eval_set"].value_counts()
print("\nÀ quel eval_set appartiennent ces commandes ?")
print(couverture)

order_id distincts dans order_products : 3214874

À quel eval_set appartiennent ces commandes ?
eval_set
prior    3214874
test           0
train          0
Name: count, dtype: int64


**Résultat (à confirmer en exécutant la cellule ci-dessus) : `order_products` ne couvre QUE les commandes
`prior`** (3 214 874)  ni `train`, ni évidemment `test`.

Conséquence concrète pour la suite :
- **Q1, Q3, Q5** (qui ont besoin du contenu des paniers) porteront uniquement sur les commandes `prior`.
- **Q2, Q4** (qui ne dépendent que de `orders` : jour/heure, délais) peuvent utiliser tout `orders` sans problème
  puisqu'elles n'ont pas besoin de `order_products`.